In [ ]:
!pip install -q -U transformers accelerate datasets huggingface_hub pandas pyarrow tqdm
!pip install -q sae-lens

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# import sys
# sys.path.insert(1, "/kaggle/input/corpus_loader")

import time
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer
from sae_lens import SAE

from corpus_loader import CorpusConfig, chunk_stream, batch_chunks

In [ ]:
MODEL_ID = "google/gemma-2-2b"
LAYER_IDX = 12
DTYPE = torch.float16

SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID_16K = f"layer_{LAYER_IDX}/width_16k/canonical"
SAE_ID_65K = f"layer_{LAYER_IDX}/width_65k/canonical"

# ---- исследуемые фичи 16k ----
SOURCE_FEATURES = [10301, 2241, 11343, 12428, 1673]

# известные триггеры сильного пика — для шага C
KNOWN_STRONG_TRIGGER = {
    10301: "usepackage",
    2241:  "{",
    11343: "www",
    12428: "://",
    1673:  " However",
}

TOP_N = 8            # сколько ближайших 65k-фич брать на каждую 16k-фичу

# ---- корпус ----
SEQ_LEN = 1024
TARGET_TOKENS = 10_000_000
BATCH_SIZE = 8
SAE_CHUNK = 1024
CTX_BEFORE, CTX_AFTER = 30, 10

SECRET_NAME = "llama-token"
OUTPUT_DIR = "/kaggle/working"
COSSIM_PATH = os.path.join(OUTPUT_DIR, "pass3_cossim.parquet")
OUT_PATH = os.path.join(OUTPUT_DIR, "pass3_contexts_65k.parquet")
FLUSH_EVERY = 50_000
MAX_RUNTIME_HOURS = 8.0

In [ ]:
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret(SECRET_NAME)
    print("secret ok:", hf_token[:7] + "...")
except Exception as e:
    print("Секрет не получен:", repr(e))
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

## cosine similarity между decoder-векторами

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def load_sae(sae_id):
    out = SAE.from_pretrained(release=SAE_RELEASE, sae_id=sae_id, device=device)
    return out[0] if isinstance(out, tuple) else out

sae16 = load_sae(SAE_ID_16K); sae16.eval()
sae65 = load_sae(SAE_ID_65K); sae65.eval()

print("16k W_dec:", tuple(sae16.W_dec.shape))
print("65k W_dec:", tuple(sae65.W_dec.shape))

assert sae16.W_dec.shape[1] == sae65.W_dec.shape[1], "d_model не совпадает!"
D16, d_model = sae16.W_dec.shape
D65, _ = sae65.W_dec.shape
assert max(SOURCE_FEATURES) < D16
print(f"OK: D16={D16}, D65={D65}, d_model={d_model}")

In [ ]:
with torch.no_grad():
    W16 = sae16.W_dec.float()
    W65 = sae65.W_dec.float()
    W16n = W16 / W16.norm(dim=1, keepdim=True)
    W65n = W65 / W65.norm(dim=1, keepdim=True)

    src = W16n[SOURCE_FEATURES]                    # (5, d_model)
    cossim = src @ W65n.T                          # (5, D65)
    print("cossim матрица:", tuple(cossim.shape))

    self_sim = (W16n[SOURCE_FEATURES] * W16n[SOURCE_FEATURES]).sum(dim=1)
    print("self-similarity (должно быть 1.0):", self_sim.cpu().numpy().round(4))
    assert torch.allclose(self_sim, torch.ones_like(self_sim), atol=1e-3)

    top_vals, top_idx = cossim.topk(TOP_N, dim=1)

rows = []
for i, f16 in enumerate(SOURCE_FEATURES):
    for rank in range(TOP_N):
        rows.append({
            "src_16k": f16,
            "rank": rank,
            "tgt_65k": int(top_idx[i, rank]),
            "cossim": float(top_vals[i, rank]),
        })
cos_df = pd.DataFrame(rows)
cos_df.to_parquet(COSSIM_PATH, index=False)

for f16 in SOURCE_FEATURES:
    s = cos_df[cos_df.src_16k == f16]
    pairs = ", ".join(f"{int(r.tgt_65k)}({r.cossim:.3f})" for _, r in s.iterrows())
    print(f"  16k#{f16:5d} -> {pairs}")

TARGET_65K = sorted(cos_df.tgt_65k.unique().tolist())
print(f"\nВсего уникальных 65k-кандидатов: {len(TARGET_65K)}")

In [ ]:
with torch.no_grad():
    g = torch.Generator(device=W16n.device).manual_seed(0)
    ridx = torch.randint(0, D16, (2000,), generator=g, device=W16n.device)
    rjdx = torch.randint(0, D65, (2000,), generator=g, device=W16n.device)
    rnd = (W16n[ridx] * W65n[rjdx]).sum(dim=1).cpu().numpy()

print("косинус случайных пар 16k<->65k:")
print(f"  median={np.median(rnd):.4f}  p95={np.percentile(rnd,95):.4f}  "
      f"p99={np.percentile(rnd,99):.4f}  max={rnd.max():.4f}")
print()
print("косинус наших топ-1 совпадений:")
top1 = cos_df[cos_df['rank']==0]
for _, r in top1.iterrows():
    z = (r.cossim - rnd.mean()) / rnd.std()
    print(f"  16k#{int(r.src_16k):5d} -> 65k#{int(r.tgt_65k):5d}: {r.cossim:.3f}  ({z:.1f} sigma от случайного)")

## прогон корпуса — на чём кандидаты реально стреляют

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map={"": 0})
model.eval()

device = next(model.parameters()).device
print("devices:", set(str(p.device) for p in model.parameters()))
print("n_layers:", len(model.layers), "| d_model:", model.config.hidden_size)

del sae16, W16, W16n
torch.cuda.empty_cache()

sae65 = sae65.to(device)
feat_tensor = torch.tensor(TARGET_65K, device=device)
print(f"следим за {len(TARGET_65K)} фичами 65k-словаря")

In [ ]:
_captured = {}

def _hook_fn(module, inputs, output):
    _captured["hidden"] = output[0] if isinstance(output, tuple) else output

target_module = model.layers[LAYER_IDX]
handle = target_module.register_forward_hook(_hook_fn)
print(f"Hook на model.layers[{LAYER_IDX}] (residual stream, post-block)")

In [ ]:
import glob

records, n_flushed, _part = [], 0, 0
PART_GLOB = os.path.join(OUTPUT_DIR, "pass3_part_*.parquet")
for p in glob.glob(PART_GLOB):
    os.remove(p)

def flush(force=False):
    global records, n_flushed, _part
    if not records or (len(records) < FLUSH_EVERY and not force):
        return
    path = os.path.join(OUTPUT_DIR, f"pass3_part_{_part:04d}.parquet")
    pd.DataFrame(records).to_parquet(path, index=False)
    n_flushed += len(records); _part += 1; records = []

In [ ]:
cfg = CorpusConfig(seq_len=SEQ_LEN, skip_tokens=0, max_tokens=TARGET_TOKENS, seed=0)
batches = batch_chunks(chunk_stream(tokenizer, cfg), BATCH_SIZE)

total_batches = TARGET_TOKENS // (BATCH_SIZE * SEQ_LEN)
start = time.time()
tokens_seen = 0

pbar = tqdm(batches, total=total_batches, desc="пасс 3", unit="batch")
with torch.no_grad():
    for n_batches, batch in enumerate(pbar):
        if (time.time() - start) / 3600 > MAX_RUNTIME_HOURS:
            print("Бюджет времени исчерпан.")
            break

        batch_gpu = batch.to(device)
        model(batch_gpu)

        hidden = _captured["hidden"]
        B, T, _ = hidden.shape
        hidden_flat = hidden.reshape(-1, hidden.shape[-1])
        batch_cpu = batch.numpy()

        for i in range(0, hidden_flat.shape[0], SAE_CHUNK):
            sub = hidden_flat[i:i + SAE_CHUNK].to(sae65.W_enc.dtype)
            z = sae65.encode(sub)
            z_sub = z[:, feat_tensor]

            rows_, cols_ = z_sub.nonzero(as_tuple=True)
            if rows_.numel():
                vals = z_sub[rows_, cols_].float().cpu().numpy()
                rows_ = rows_.cpu().numpy(); cols_ = cols_.cpu().numpy()
                for row, col, val in zip(rows_, cols_, vals):
                    flat_pos = i + int(row)
                    b, t = flat_pos // T, flat_pos % T
                    lo = max(0, t - CTX_BEFORE)
                    hi = min(T, t + CTX_AFTER + 1)
                    ctx_ids = batch_cpu[b, lo:hi]

                    records.append({
                        "feature_65k": TARGET_65K[int(col)],
                        "value": float(val),
                        "token_id": int(batch_cpu[b, t]),
                        "token": tokenizer.decode([int(batch_cpu[b, t])]),
                        "context": tokenizer.decode(ctx_ids),
                        "ctx_token_ids": ctx_ids.tolist(),
                        "target_idx_in_ctx_tokens": int(t - lo),
                        "global_chunk": n_batches * BATCH_SIZE + b,
                        "pos_in_chunk": int(t),
                    })
            del z, z_sub, sub

        flush()
        del hidden, hidden_flat
        _captured.clear()

        tokens_seen += B * T
        if n_batches % 20 == 0:
            pbar.set_postfix(tokens=f"{tokens_seen/1e6:.1f}M", records=f"{n_flushed+len(records):,}")

flush(force=True)
parts = sorted(glob.glob(PART_GLOB))
if parts:
    pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True).to_parquet(OUT_PATH, index=False)
    for p in parts:
        os.remove(p)
    print(f"Склеено {len(parts)} частей -> {OUT_PATH}")
else:
    print("ВНИМАНИЕ: ни одной записи!")

print(f"Готово. tokens_seen={tokens_seen:,}, записей={n_flushed:,}")
handle.remove()

## Шаг C: расщепились ли триггеры?

In [ ]:
df3 = pd.read_parquet(OUT_PATH)
print(f"записей: {len(df3):,}\n")

bad = 0
for _, r in df3.sample(min(2000, len(df3)), random_state=0).iterrows():
    if r.ctx_token_ids[r.target_idx_in_ctx_tokens] != r.token_id:
        bad += 1
print(f"проверка индекса: несовпадений {bad}/2000 (должно быть 0)")

In [ ]:
for f16 in SOURCE_FEATURES:
    trig = KNOWN_STRONG_TRIGGER[f16]
    print("=" * 78)
    print(f"16k#{f16} | сильный триггер в 16k: {trig!r}")
    print("-" * 78)
    cands = cos_df[cos_df.src_16k == f16].sort_values("rank")
    for _, c in cands.iterrows():
        f65 = int(c.tgt_65k)
        s = df3[df3.feature_65k == f65]
        if len(s) == 0:
            print(f"  65k#{f65:5d} cos={c.cossim:.3f} | не сработала ни разу")
            continue
        vc = s.token.value_counts()
        top = ", ".join(f"{t!r}:{n}" for t, n in vc.head(4).items())
        trig_share = (s.token == trig).mean()
        print(f"  65k#{f65:5d} cos={c.cossim:.3f} n={len(s):5d} уник={s.token.nunique():4d} "
              f"| {trig!r} = {trig_share:5.1%}")
        print(f"           топ: {top}")
    print()

In [ ]:
print("=== ВЕРДИКТ ПО КАЖДОЙ ФИЧЕ ===\n")
verdict = []
for f16 in SOURCE_FEATURES:
    trig = KNOWN_STRONG_TRIGGER[f16]
    cands = cos_df[cos_df.src_16k == f16]
    best_pure, best_f65, best_n = 0.0, None, 0
    for f65 in cands.tgt_65k:
        s = df3[df3.feature_65k == int(f65)]
        if len(s) < 50:
            continue
        share = (s.token == trig).mean()
        if share > best_pure:
            best_pure, best_f65, best_n = share, int(f65), len(s)

    split = best_pure > 0.8
    verdict.append({"src_16k": f16, "trigger": trig, "best_65k": best_f65,
                    "trigger_purity": best_pure, "n": best_n, "split_confirmed": split})
    mark = "ДА " if split else "НЕТ"
    print(f"  {mark} 16k#{f16:5d} ({trig!r}): лучший кандидат 65k#{best_f65} — "
          f"{best_pure:.1%} срабатываний на триггере (n={best_n})")

vdf = pd.DataFrame(verdict)
print(f"\nРасщепление подтверждено: {int(vdf.split_confirmed.sum())} / {len(vdf)}")
vdf.to_parquet(os.path.join(OUTPUT_DIR, "pass3_verdict.parquet"), index=False)